In [1]:
import torch
import torch.nn.functional as F
import requests
import random
from torch.utils.data import DataLoader, TensorDataset
import tqdm

/home/xerneas/jupyter-env/.venv/lib/python3.12/site-packages/torch/_subclasses/functional_tensor.py:362: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [2]:
url = "https://raw.githubusercontent.com/karpathy/makemore/master/names.txt"
words = requests.get(url).text.splitlines()

random.shuffle(words)
print(words[:10])
len(words)

['chasya', 'aseer', 'andi', 'siyah', 'jibril', 'riki', 'haji', 'vallerie', 'celestia', 'xiclaly']


32033

In [3]:
if device := torch.device("mps" if torch.backends.mps.is_available() else "cpu"):
    print(f"Using device: {device}")

block_size = 7
epochs = 200
lr = 0.001



Using device: cpu


In [4]:
chars = ['.'] + sorted(set(''.join(words)))
def stoi(s):
    return {c: i for i, c in enumerate(chars)}[s]
def itos(i):
    return {i: c for i, c in enumerate(chars)}[i]
print(itos(1), itos(2), itos(3), itos(0))
print(stoi('.'), stoi('a'), stoi('b'), stoi('c'))

a b c .
0 1 2 3


In [5]:
def build_dataset(words):
    X, Y = [], []
    for w in words:
        context = [0] * block_size
        for ch in w + '.':
            ix = stoi(ch)
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [6]:
X, Y = build_dataset(words)
n = X.shape[0]
Xtr, Ytr = X[:int(n*0.9)], Y[:int(n*0.9)]
Xdev, Ydev = X[int(n*0.9):], Y[int(n*0.9):]
Xtr, Ytr = [x.to(device) for x in (Xtr, Ytr)]
Xdev, Ydev = [x.to(device) for x in (Xdev, Ydev)]
print(X.shape, Y.shape)
print(Xtr.shape, Ytr.shape)
print(Xdev.shape, Ydev.shape)

torch.Size([228146, 7]) torch.Size([228146])
torch.Size([205331, 7]) torch.Size([205331])
torch.Size([22815, 7]) torch.Size([22815])


In [7]:
train_ds = torch.utils.data.TensorDataset(Xtr.cpu(), Ytr.cpu())

train_dl = DataLoader(
    dataset = train_ds,
    batch_size = 1024,
    pin_memory = False,
    shuffle = True,
)

In [8]:
class MLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.C = torch.nn.Parameter(torch.randn((27, 10)))
        self.W1 = torch.nn.Parameter(torch.randn((70, 200))/(5/3) * (70**0.5))
        self.b1 = torch.nn.Parameter(torch.randn(200)*0.1)
        self.W2 = torch.nn.Parameter(torch.randn((200, 27))*0.01)
        self.b2 = torch.nn.Parameter(torch.zeros(27))

    def forward(self, Xb):
        emb = self.C[Xb]
        emb = emb.view(-1, 70)
        h = torch.tanh(emb @ self.W1 + self.b1)
        logits = h @ self.W2 + self.b2
        return logits


In [9]:
model = MLP().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.9999)



In [10]:
for e in range(epochs):
    for Xb, Yb in train_dl:
        Xb, Yb = Xb.to(device), Yb.to(device)
        logits = model(Xb)
        loss = F.cross_entropy(logits, Yb)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        scheduler.step()
    if e % 10 == 0:
        print(e, loss.item(), scheduler.get_last_lr()[0])


0 2.323240280151367 0.0009800996732739183
10 2.1474790573120117 0.0008016276499971287
20 2.2359471321105957 0.0006556546306084981
30 2.182378053665161 0.0005362626833541804
40 2.1943469047546387 0.0004386115069321365
50 2.0747053623199463 0.0003587421985247839
60 2.218158006668091 0.0002934167548465804
70 2.1039814949035645 0.00023998679937495656
80 2.1533825397491455 0.00019628621379971938
90 2.167055368423462 0.00016054332083337712
100 2.1380412578582764 0.00013130905816191102
110 2.2176787853240967 0.00010739823161664415
120 2.0967469215393066 8.784146589612893e-05
130 2.122453451156616 7.184590485924672e-05
140 2.2591795921325684 5.876306812943797e-05
150 2.1716160774230957 4.806256087594607e-05
160 2.27864408493042 3.9310570933187996e-05
170 2.22998309135437 3.2152281504138196e-05
180 2.2041008472442627 2.6297486436366887e-05
190 2.1929166316986084 2.1508824895735396e-05


In [11]:
ix = torch.randint(0, Xdev.shape[0], (Xdev.shape[0],), device=device)
Xb, Yb = Xdev[ix], Ydev[ix]
logits = model(Xb)
loss = F.cross_entropy(logits, Yb)
print(loss.item())

2.2208824157714844


In [17]:
for i in range(10):
    out = []
    context = [0] * block_size
    while True:
        logits = model(torch.tensor([context]))
        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1).item()
        context = context[1:] + [ix]
        out.append(itos(ix))
        if ix == 0:
            break
    print(''.join(out))

thaila.
kaiffianle.
haege.
cassya.
anteyda.
macky.
zahdri.
ihcar.
adri.
kar.
